# Retrieval backend trên Kaggle — mapping, embedding, OCR Elasticsearch, foreground logs
Bật Internet; chọn GPU nếu chạy visual encoder. Notebook tải dữ liệu ASR/OCR/caption gốc, map/keyframe, visual index và embedding ASR/caption theo từng video trong cùng cây model (`asr/<DATASET>/<video_id>/<segment_number>.npy` và `caption/<DATASET>/<video_id>/<segment_number>.npy`). Mỗi segment có vector riêng được ghép với JSON gốc theo ordinal; không embed lại ASR/caption. Notebook không tự chạy các model extraction. Điền nguồn thật trong cell cấu hình. Chạy cell cuối liên tục để giữ supervisor và xem log; Kaggle vẫn có thể kết thúc phiên theo quota/session limits.

Nguồn: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [Elasticsearch archive / non-root](https://www.elastic.co/docs/deploy-manage/deploy/self-managed/install-elasticsearch-from-archive-on-linux-macos).


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
WORK = Path('/kaggle/working')
REPO = WORK / 'event-retrieval-system'
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
# Chọn branch/commit chứa thay đổi mới, hoặc attach repo vào /kaggle/input.
REPO_REF = None
DATA = REPO / 'data'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
DATASETS = ['v3c']
QWEN_MODEL = 'Qwen/Qwen3-VL-Embedding-2B'
SEGMENT_BATCH_SIZE = 8
SEGMENT_SOURCES = ['asr', 'caption']  # Phải khớp những source đã embed.
# Mỗi source có destination tương đối dưới data; không tự đoán cấu trúc archive.
# kind: local | url | gdrive | huggingface | kaggle
SOURCES = [
    # {'kind':'local', 'source':'/kaggle/input/keyframe-scenes/V3C', 'destination':'scenes/V3C'},
    # {'kind':'url', 'source':'https://your-host/scenes.zip', 'destination':'scenes', 'archive':True},
    # {'kind':'local', 'source':'/kaggle/input/my-data/keyframes', 'destination':'keyframes'},
    # {'kind':'local', 'source':'/kaggle/input/my-data/map_keyframes', 'destination':'map_keyframes'},
    # {'kind':'url', 'source':'https://your-host/captions.zip', 'destination':'captions', 'archive':True},
    # {'kind':'gdrive', 'source':'FILE_ID_OR_URL', 'destination':'asr', 'archive':True},
    # {'kind':'huggingface', 'source':'owner/dataset', 'destination':'.', 'allow_patterns':['ocr/**','asr/**','captions/**']},
    # {'kind':'kaggle', 'source':'owner/dataset', 'destination':'.'},
    # {'kind':'local', 'source':'/kaggle/input/ers-qwen-embeddings/embeddings_qwen3_vl_embedding_2b_vllm', 'destination':'embeddings/embeddings_qwen3_vl_embedding_2b_vllm'},
    # {'kind':'gdrive_folder', 'source':'ASR_FOLDER_ID', 'destination':'asr/V3C'},
    # {'kind':'gdrive_folder', 'source':'CAPTION_FOLDER_ID', 'destination':'captions/V3C'},
    # {'kind':'gdrive_folder', 'source':'OCR_FOLDER_ID', 'destination':'ocr/V3C'},
    # {'kind':'gdrive_folder', 'source':'MAP_FOLDER_ID', 'destination':'map_keyframes/V3C'},
    # {'kind':'gdrive_folder', 'source':'SEGMENT_EMBEDDINGS_FOLDER_ID', 'destination':'embeddings/embeddings_qwen3_vl_embedding_2b_vllm'},
    # {'kind':'gdrive_folder', 'source':'VISUAL_EMBEDDINGS_FOLDER_ID', 'destination':'embeddings/embeddings_qwen3_vl_embedding_2b_vllm/V3C', 'extract_zips':True},
    # {'kind':'gdrive_folder', 'source':'KEYFRAMES_FOLDER_ID', 'destination':'keyframes/V3C'},
    # {'kind':'gdrive_folder', 'source':'VIDEOS_FOLDER_ID', 'destination':'videos/V3C'},
]
# Archives must contain files relative to destination; inspect downloaded layout.
# Visual and segment embeddings use Qwen vLLM.
VISUAL_ENGINE = 'vllm'
VLLM_COMMAND = 'auto'  # Install isolated runtime; use None for an existing remote endpoint.
# Example: ['python','-m','vllm.entrypoints.openai.api_server','--model',
# 'Qwen/Qwen3-VL-Embedding-2B','--runner','pooling','--port','8002']
# Obtain optional NGROK_AUTHTOKEN from Kaggle Secrets below. Never paste credentials here.


In [ ]:
import urllib.request
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
if REPO_REF:
    subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_REF], check=True)
BACKEND = REPO / 'backend'
assert (BACKEND/'scripts/build_segment_data.py').exists(), 'Use the branch containing new search code'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy','pandas','faiss-cpu','fastapi','uvicorn','python-multipart','httpx','PyYAML',
    'python-dotenv','elasticsearch>=8.18,<9','sentence-transformers','Pillow','huggingface_hub',
    'kagglehub','gdown','pyngrok','open_clip_torch','timm','einops','uv'], check=True)
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))

rclone = shutil.which('rclone')
if not rclone:
    import zipfile
    archive=WORK/'rclone-v1.75.2-linux-amd64.zip'
    urllib.request.urlretrieve('https://github.com/rclone/rclone/releases/download/v1.75.2/rclone-v1.75.2-linux-amd64.zip',archive)
    with zipfile.ZipFile(archive) as z:
        member=next(x for x in z.namelist() if x.endswith('/rclone'))
        rclone_bin=WORK/'rclone'; rclone_bin.write_bytes(z.read(member)); rclone_bin.chmod(0o755)
    rclone=str(rclone_bin)
RCLONE_CONFIG = WORK/'rclone.conf'
RCLONE_CONFIG_READY = False
try:
    from kaggle_secrets import UserSecretsClient
    config_text=UserSecretsClient().get_secret('RCLONE_CONFIG')
except Exception:
    config_text=None
if config_text:
    RCLONE_CONFIG.write_text(config_text,encoding='utf-8'); RCLONE_CONFIG_READY=True
elif RCLONE_CONFIG_SOURCE.is_file():
    shutil.copy2(RCLONE_CONFIG_SOURCE,RCLONE_CONFIG); RCLONE_CONFIG_READY=True
if RCLONE_CONFIG_READY:
    RCLONE_CONFIG.chmod(0o600)
    subprocess.run([rclone,'--config',str(RCLONE_CONFIG),'listremotes'],check=True)

In [ ]:
import urllib.request, zipfile, tarfile, tempfile
from huggingface_hub import snapshot_download

def destination(relative):
    path = (DATA / relative).resolve()
    if not path.is_relative_to(DATA.resolve()):
        raise ValueError('Destination outside data root')
    path.mkdir(parents=True, exist_ok=True)
    return path

def unpack(archive, target):
    # Reject traversal and links; archives are untrusted input.
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                if not (target/entry.filename).resolve().is_relative_to(target.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Unsafe archive member')
            z.extractall(target)
    else:
        with tarfile.open(archive) as t:
            for entry in t.getmembers():
                if not (target/entry.name).resolve().is_relative_to(target.resolve()) or entry.issym() or entry.islnk() or not (entry.isfile() or entry.isdir()):
                    raise ValueError('Unsafe archive member')
            t.extractall(target)

for source in SOURCES:
    target = destination(source['destination'])
    kind, origin = source['kind'], source['source']
    print('Downloading', kind, '→', target, flush=True)
    if kind == 'huggingface':
        snapshot_download(repo_id=origin, repo_type='dataset', local_dir=target,
                          allow_patterns=source.get('allow_patterns'))
    elif kind == 'gdrive_folder':
        if RCLONE_CONFIG_READY:
            listing=subprocess.run([rclone,'--config',str(RCLONE_CONFIG),'--drive-root-folder-id',origin,
                                   'lsf','iamtro:','--recursive','--files-only','--format','p'],
                                  check=True,capture_output=True,text=True)
            files=[x.strip() for x in listing.stdout.splitlines() if x.strip()]
            for relative in files:
                rel=Path(relative)
                if rel.is_absolute() or '..' in rel.parts:
                    raise ValueError('Unsafe Google Drive relative path')
                destination_path=target/rel
                destination_path.parent.mkdir(parents=True,exist_ok=True)
                subprocess.run([rclone,'--config',str(RCLONE_CONFIG),'--drive-root-folder-id',origin,
                                'copyto',f'iamtro:{relative}',str(destination_path)],check=True)
            if not files:
                raise RuntimeError('Google Drive folder is empty')
        else:
            import gdown
            folder_url = origin if origin.startswith('http') else 'https://drive.google.com/drive/folders/' + origin
            downloaded = gdown.download_folder(folder_url, output=str(target), quiet=False, remaining_ok=True)
            if not downloaded:
                raise RuntimeError('Google Drive folder download failed; configure Kaggle Secret RCLONE_CONFIG for private folders')
    elif kind == 'kaggle':
        import kagglehub
        cached = Path(kagglehub.dataset_download(origin))
        shutil.copytree(cached, target, dirs_exist_ok=True)
    elif kind == 'local' and Path(origin).is_dir():
        shutil.copytree(origin, target, dirs_exist_ok=True)
    else:
        with tempfile.TemporaryDirectory() as tmp:
            archive = Path(tmp)/'download'
            if kind == 'local':
                shutil.copyfile(origin, archive)
            elif kind == 'url':
                urllib.request.urlretrieve(origin, archive)
            elif kind == 'gdrive':
                import gdown
                if not gdown.download(origin if origin.startswith('http') else 'https://drive.google.com/uc?id='+origin, str(archive), fuzzy=True, quiet=False):
                    raise RuntimeError('Drive download failed')
            else:
                raise ValueError('Unknown source kind')
            if source.get('archive', False):
                unpack(archive, target)
            else:
                filename = source.get('filename')
                if not filename or Path(filename).name != filename:
                    raise ValueError('Non-archive source requires a plain filename')
                shutil.copyfile(archive, target/filename)
    if source.get('extract_zips', False):
        archives = sorted(target.rglob('*.zip'))
        if not archives:
            raise FileNotFoundError(f'No per-video ZIPs found under {target}')
        for archive_path in archives:
            unpack(archive_path, target)
            archive_path.unlink()
        print(f'Expanded {len(archives)} embedding ZIPs under {target}', flush=True)
print('Data roots:', [p.name for p in DATA.iterdir()])


## Cấu hình dataset và build mapping/index
Cần keyframes, map_keyframes, ASR JSON, caption JSON, OCR JSON và visual FAISS index/metadata tương ứng engine. Caption/ASR cùng visual dùng Qwen3-VL-Embedding-2B qua vLLM. Không có dữ liệu thật thì dừng rõ lỗi; không fallback sang mock. Text model chung cho ASR/caption có thể đổi; đổi model phải rebuild cả hai index.


In [ ]:
if VISUAL_ENGINE == 'vllm' and VLLM_COMMAND == 'auto':
    VLLM_VENV = WORK / 'ers-vllm-venv'
    if not (VLLM_VENV/'bin/vllm').exists():
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(VLLM_VENV)], check=True)
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(VLLM_VENV/'bin/python'),
                        '--torch-backend=auto', '-r', str(BACKEND/'requirements-vllm.txt')], check=True)
    VLLM_COMMAND = [str(VLLM_VENV/'bin/vllm'), 'serve', 'Qwen/Qwen3-VL-Embedding-2B',
                    '--runner', 'pooling', '--max-model-len', '8192', '--host', '127.0.0.1', '--port', '8002']
    os.environ['QWEN_VLLM_URL'] = 'http://127.0.0.1:8002/v1'
elif VLLM_COMMAND == 'auto':
    VLLM_COMMAND = None


In [ ]:
import yaml
APP_CONFIG = BACKEND / 'configs/app.yaml'
cfg = yaml.safe_load(APP_CONFIG.read_text())
for key in list(cfg['datasets']['available']):
    if key not in DATASETS:
        del cfg['datasets']['available'][key]
cfg['datasets']['default'] = DATASETS[0]
cfg['segment_data'] = {'data_root': str(DATA)}
cfg['asr'] = None
# No translation API key is required for the notebook's default multilingual search.
cfg.setdefault('translate', {})['enabled_default'] = False
for dataset, spec in cfg['datasets']['available'].items():
    spec.update(root=str(DATA), keyframes_root=str(DATA/'keyframes'),
                videos_root=str(DATA/'videos'), map_keyframes_root=str(DATA/'map_keyframes'))
    # Paths/engine suffixes must match the downloaded visual index artifacts.
    for model in cfg['semantic']['models']:
        model.setdefault('model_extra', {})['engine'] = VISUAL_ENGINE
# Fail visibly before hosting if downloaded visual artifacts are absent.
for dataset, spec in cfg['datasets']['available'].items():
    for model in cfg['semantic']['models']:
        if not model.get('enabled', True):
            continue
        override = spec['semantic_models'].get(model['model_key'], {})
        directory = override.get('index_dir')
        if directory:
            directory = Path(directory)
            if not directory.is_absolute():
                directory = BACKEND / directory
            if not (directory/'keyframe.faiss').is_file():
                # Reuse the repository's visual index builder if only raw embeddings were downloaded.
                from src.retrieval.indexer.faiss.build_pipeline import BuildFaissIndexPipeline
                embeddings = DATA/'embeddings'/f'embeddings_qwen3_vl_embedding_2b_{VISUAL_ENGINE}'
                assert embeddings.exists(), f'Download a visual index or raw embeddings into {embeddings}'
                indexing = yaml.safe_load((BACKEND/'configs/indexing.yaml').read_text())
                build = {**indexing, **indexing['datasets']['available'][dataset],
                         'keyframes_root':str(DATA/'keyframes'),
                         'map_keyframes_root':str(DATA/'map_keyframes'),
                         'embeddings_root':str(embeddings), 'collection':dataset,
                         'index':{**indexing['index'], 'output_dir':str(directory)}}
                print('Building visual FAISS index:', dataset, directory, flush=True)
                BuildFaissIndexPipeline(build).run()
            for filename in ('keyframe.faiss', 'metadata.csv'):
                assert (directory/filename).is_file(), f'Download visual artifact: {directory/filename}'
            cache_path = directory/'vectors_fp32.npy'
            if not cache_path.is_file():
                subprocess.run([sys.executable,'-u','-m','src.retrieval.indexer.faiss.vector_cache',
                                '--index-path',str(directory/'keyframe.faiss'),'--output',str(cache_path),
                                '--dtype','float32','--batch-size','50000'],cwd=BACKEND,check=True)
# Write runtime config (not committed); scripts and API use it through CONFIG_PATH.
RUNTIME_CONFIG = BACKEND/'configs/kaggle.runtime.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
os.environ['CONFIG_PATH'] = str(RUNTIME_CONFIG)
os.environ['QWEN_EMBEDDING_ENGINE'] = 'vllm'
os.environ.setdefault('QWEN_VLLM_URL', 'http://127.0.0.1:8002/v1')
if VISUAL_ENGINE != 'vllm':
    raise RuntimeError('Visual and text indexes must use Qwen vLLM')
from src.retrieval.segment_data import build_maps, load_segment_files, SegmentMaps
for dataset in DATASETS:
    build_maps(DATA,dataset.lower())
    for folder in ('captions','ocr'):
        assert list((DATA/folder/dataset.upper()).glob('*.json')), f'Missing {folder}/{dataset}'
    if 'asr' in SEGMENT_SOURCES:
        assert list((DATA/'asr'/dataset.upper()).glob('*.json')), f'Missing asr/{dataset}'
    assert list((DATA/'map_keyframes'/dataset.upper()).glob('*.csv')), f'Missing map_keyframes/{dataset}'
    for source in SEGMENT_SOURCES:
        segment_dir = DATA/'asr' if source=='asr' else DATA/'captions'
        json_files = list((segment_dir/dataset.upper()).glob('*.json'))
        assert json_files, f'Missing {source} JSON for {dataset}'
        source_files=dict(load_segment_files(DATA,dataset.lower(),source))
        if source=='caption':
            all_records=[record for records in source_files.values() for record in records]
            maps=SegmentMaps(DATA,dataset.lower(),'caption',all_records)
            source_files={video_id:[record for record in records if maps.forward.get(record['segment_id'])]
                          for video_id,records in source_files.items()}
        for json_path in json_files:
            records=source_files.get(json_path.stem,[])
            vector_dir=DATA/'embeddings'/'embeddings_qwen3_vl_embedding_2b_vllm'/source/dataset.upper()/json_path.stem
            expected={f"{int(record['segment_id'].rsplit(':',1)[-1]):06d}.npy" for record in records}
            actual={path.name for path in vector_dir.glob('*.npy')} if vector_dir.is_dir() else set()
            assert actual==expected, f'Missing/mismatched per-segment {source} vectors for {json_path.stem}: missing={sorted(expected-actual)[:10]} extra={sorted(actual-expected)[:10]}'
print('Inputs ready; the host supervisor will start Qwen vLLM, build ASR/caption FAISS, seed OCR, and start the API.')

## Cài Elasticsearch local (không cần Docker)
Tải archive chính thức và kiểm checksum. Cell cuối chạy Elasticsearch bằng tài khoản không phải root, seed OCR với `--recreate`, và giám sát toàn bộ tiến trình. Elasticsearch chỉ bind localhost.


In [ ]:
import hashlib
ES_VERSION = '8.18.0'
ES_HOME = WORK / f'elasticsearch-{ES_VERSION}'
if not ES_HOME.exists():
    name = f'elasticsearch-{ES_VERSION}-linux-x86_64.tar.gz'
    archive = WORK/name
    url = 'https://artifacts.elastic.co/downloads/elasticsearch/'+name
    urllib.request.urlretrieve(url, archive)
    expected = urllib.request.urlopen(url+'.sha512').read().decode().split()[0]
    digest = hashlib.sha512()
    with archive.open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    assert digest.hexdigest() == expected, 'Elasticsearch checksum mismatch'
    with tarfile.open(archive) as bundle:
        bundle.extractall(WORK, filter='data')
(ES_HOME/'config/elasticsearch.yml').write_text("""cluster.name: ers-kaggle
node.name: notebook
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
""")
print('ES installed:', ES_HOME)


## Chạy foreground — giữ cell này đang chạy
Log FastAPI/access, lỗi startup/search, log ES và heartbeat đều hiện trong cell. Dừng cell sẽ dọn các tiến trình do supervisor tạo. Log không ngăn Kaggle thu hồi phiên/quota hay OOM; cần lưu output/index ra nguồn bền vững. Backend không tự đăng nhập Codex trên Kaggle: agent cần CLI/login hợp lệ ở host chạy bridge; các endpoint search thường vẫn hoạt động độc lập.

Nếu dùng vLLM, cài theo `backend/requirements-vllm.txt` trong môi trường tương thích rồi điền `VLLM_COMMAND`, hoặc cung cấp endpoint qua `QWEN_VLLM_URL`. Không dùng index được tạo bởi engine khác.


In [ ]:
from scripts.kaggle_host import serve
ngrok_token = None
try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
except Exception:
    print('No ngrok secret: API available inside notebook at http://127.0.0.1:8000', flush=True)
# Intentionally await the server IN THIS CELL: no detached API subprocess, no reload.
await serve({'backend':str(BACKEND), 'app_config':str(RUNTIME_CONFIG),
             'es_home':str(ES_HOME), 'datasets':DATASETS, 'port':8000,
             'vllm_command':VLLM_COMMAND, 'ngrok_token':ngrok_token,
             'build_segment_indexes':True, 'data_root':str(DATA),
             'segment_embeddings_root':str(DATA/'embeddings'/'embeddings_qwen3_vl_embedding_2b_vllm'),
             'segment_model':QWEN_MODEL, 'segment_batch_size':SEGMENT_BATCH_SIZE,
             'segment_sources':SEGMENT_SOURCES})
